# Plot images

This notebook contains code that will allow you to download and display images corresponding to wells of interest.

In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tifffile
from sh import aws
import re
import os

In [5]:
# Get image URLs and make output directories for images
img_urls = pd.read_csv("../../inputs/load_data.csv")
os.makedirs("../../outputs/images/tiff", exist_ok=True)   
os.makedirs("../../outputs/images/png", exist_ok=True)

In [7]:
# Plot images
well = "A08"
site = 5
base_url = img_urls.query("Metadata_Well == @well and Metadata_Site == @site")["URL_OrigRNA"].values[0]

# Download images
files = []
for ch in range(1, 6):
    img_url = re.sub(r'-ch\d+', f'-ch{ch}', base_url)
    tiff_path = "../../outputs/images/tiff/" + os.path.basename(img_url)

    if not os.path.exists(tiff_path):
        aws("s3", "cp", img_url, "../../outputs/images/tiff", "--no-sign-request")

    files.append(tiff_path)


# Create image
names = ["Mitochondria", "AGP", "RNA", "ER", "DNA"]

colors = [
    np.array([1, 0, 0], dtype=np.float32),      # Red
    np.array([1, 1, 0], dtype=np.float32),      # Yellow
    np.array([0.6, 0, 0.8], dtype=np.float32),  # Purple
    np.array([0, 1, 0], dtype=np.float32),      # Green
    np.array([0, 0, 1], dtype=np.float32),      # Blue
]

def clip_norm(img, p=99):
    img = img.astype(np.float32)
    hi = np.percentile(img, p)
    if hi <= 0:
        return np.zeros_like(img, dtype=np.float32)
    return np.clip(img, 0, hi) / hi

# Load + normalize channels
chs = [clip_norm(tifffile.imread(f), p=99) for f in files]

# Colorize each channel (H,W,3)
colored_chs = [ch[..., None] * col for ch, col in zip(chs, colors)]

# Merge by addition, then rescale composite to avoid saturation blobs
merged = np.zeros_like(colored_chs[0], dtype=np.float32)
for cc in colored_chs:
    merged += cc

comp_hi = np.percentile(merged, 99.9)  # tweak (99.5–99.99) if desired
if comp_hi > 0:
    merged = merged / comp_hi
merged = np.clip(merged, 0, 1)

# Plot 5 singles + merged in a 2x3 grid
fig, axes = plt.subplots(2, 3, figsize=(12, 8))
axes = axes.ravel()

for i in range(5):
    axes[i].imshow(np.clip(colored_chs[i], 0, 1))
    axes[i].set_title(names[i])
    axes[i].axis("off")

axes[5].imshow(merged)
axes[5].set_title("Merged")
axes[5].axis("off")

plt.tight_layout()
fig.savefig(f"../../outputs/images/png/{well}_{site}.png", dpi=300, bbox_inches="tight")
plt.close(fig)

**Exercise** 
- Modify code to download and display images corresponding to an other well of interest.